In [1]:
%pip install "sagemaker<3" -q
!pip install pyathena awswrangler  --quiet
!pip install 'boto3>1.17.21' -q

Note: you may need to restart the kernel to use updated packages.


# Set up Athena Pneumonia DB
This notebook is purely for a user to register the pneumonia db in thier Glue Catalog

In [2]:
import boto3
import sagemaker
import pandas as pd
from pyathena import connect
import awswrangler as wr

sess = sagemaker.Session()
default_bucket = sess.default_bucket()
region = boto3.Session().region_name
bucket = "pneumonia-data-set-group-4"

# Athena staging directory (uses YOUR default bucket for query results)
s3_staging_dir = f"s3://{default_bucket}/athena/staging"

# Connect to Athena
conn = connect(region_name=region, s3_staging_dir=s3_staging_dir)

print(f"Region: {region}")
print(f"Data bucket: {bucket}")
print(f"Staging dir: {s3_staging_dir}")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


Region: us-east-1
Data bucket: pneumonia-data-set-group-4
Staging dir: s3://sagemaker-us-east-1-455131748909/athena/staging


In [3]:
database_name = "pneumonia_db"

### Create the Database if it doesn't exist

In [6]:
statement = f"CREATE DATABASE IF NOT EXISTS {database_name}"
print(statement)
pd.read_sql(statement, conn)
print("Database created!")

CREATE DATABASE IF NOT EXISTS pneumonia_db


/tmp/ipykernel_829/2136708136.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(statement, conn)


Database created!


### Drop the DB table if it exists

In [4]:
drop_statement = f"""
DROP TABLE IF EXISTS {database_name}.image_metadata
"""
pd.read_sql(drop_statement, conn)

/tmp/ipykernel_70689/2168075215.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(drop_statement, conn)


""


## Create table

In [8]:
statement = f"""
CREATE EXTERNAL TABLE IF NOT EXISTS {database_name}.image_metadata (
    image_id              STRING,
    raw_s3_key            STRING,
    preprocessed_s3_key   STRING,
    label                 STRING,
    label_int             TINYINT,
    split                 STRING,
    source                STRING,
    file_type             STRING,
    pixel_mean            DOUBLE,
    pixel_std             DOUBLE,
    img_height            INT,
    img_width             INT,
    event_time            STRING
)
ROW FORMAT DELIMITED
FIELDS TERMINATED BY ','
LOCATION 's3://{bucket}/pneumonia-project/metadata/'
TBLPROPERTIES ('skip.header.line.count'='1')
"""
pd.read_sql(statement, conn)
print("Table created!")

/tmp/ipykernel_829/1396329963.py:22: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(statement, conn)


Table created!


### Validate the table was created and query it

In [9]:
import awswrangler as wr

In [5]:
query = f"SELECT * FROM {database_name}.image_metadata LIMIT 100"

df_meta = wr.athena.read_sql_query(
    query,
    database=database_name
)

2026-06-07 22:50:06,874	WARNING services.py:2137 -- WARNING: The object store is using /tmp instead of /dev/shm because /dev/shm has only 1908387840 bytes available. This will harm performance! You may be able to free up space by deleting files in /dev/shm. If you are inside a Docker container, you can increase /dev/shm size by passing '--shm-size=4.14gb' to 'docker run' (or add it to the run_options list in a Ray cluster config). Make sure to set this to more than 30% of available RAM.


2026-06-07 22:50:07,035	INFO worker.py:2007 -- Started a local Ray instance.


/opt/conda/lib/python3.12/site-packages/ray/_private/worker.py:2046: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


In [6]:
try:
    if not df_meta.empty:
        print(f'Successfully pulled from {database_name}')
        
    else:
        print('FAILED - Pulled in data is blank, please re-try setup')
except:
    print('DB and Table set up were not successfull, please try again')

Successfully pulled from pneumonia_db


In [7]:
df_meta.head(2)

,image_id,raw_s3_key,preprocessed_s3_key,label,label_int,source,file_type,pixel_mean,pixel_std,img_height,img_width,event_time
0,IM-0001-0001,raw-images/chest-xray/test/NORMAL/IM-0001-0001...,preprocessed-images/NORMAL/IM-0001-0001.png,NORMAL,0,chest_xray,jpeg,122.9340,60.6187,512,512,2026-06-07T21:26:28Z
1,IM-0003-0001,raw-images/chest-xray/test/NORMAL/IM-0003-0001...,preprocessed-images/NORMAL/IM-0003-0001.png,NORMAL,0,chest_xray,jpeg,130.1997,60.4193,512,512,2026-06-07T21:26:28Z
